# Aula 13 · Regressão linear

Esta aula apresenta o [capítulo 13 do site](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/). A ideia central: **com medições que têm ruído, não se força a curva a passar por todos os pontos — procura-se a que erra menos**. Para a reta, isso é um sistema linear 2 × 2, as equações normais.

**Ao fim da aula você consegue:**

1. deduzir as equações normais minimizando a soma dos quadrados dos resíduos;
2. ajustar uma reta a dados e medir a qualidade dela com o $R^2$;
3. ler o gráfico dos resíduos e reconhecer quando a reta não serve;
4. prever com a reta sem cair na extrapolação, e sem confundir correlação com causa.

**Roteiro:** 🧩 · 1. dados com ruído · 2. 🧑‍🏫 mínimos quadrados · 3. quão boa · 4. os resíduos · 5. confira · 6. outra área · 🎯 prática · 🧩 o CO₂ · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

# --- dados desta aula (baixados do site, se ainda não estiverem aqui) ---
import os
import urllib.request

for ARQUIVO in ["latencia_servidor.csv", "co2_mauna_loa.csv", "recordes_100m.csv"]:
    if not os.path.exists(ARQUIVO):
        urllib.request.urlretrieve("https://lacouth.github.io/metodos_telecom-site/dados/" + ARQUIVO, ARQUIVO)
    print(ARQUIVO, "pronto")

## 🧩 O problema da aula

> **Clima — quando passamos de 450 ppm?**
>
> *Os acordos climáticos usam limites de concentração de CO₂ na atmosfera como
> referência. Uma jornalista quer escrever uma matéria e pergunta a um grupo de
> estudantes: "**no ritmo atual, em que ano o CO₂ passa de 450 ppm?**" Eles têm as
> médias anuais de Mauna Loa, de 1960 a 2020.*

No fim da aula, você responde com uma reta — e descobre, pelos resíduos, que a
resposta está otimista demais.

## 1. Dados com ruído

A latência de um servidor web (ms) medida com vários números de usuários.

📖 [capítulo 13 · Dados com ruído](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#dados-com-ruido)

In [ ]:
# 📦 dados prontos — só rode esta célula
dados = np.loadtxt("latencia_servidor.csv", delimiter=",", skiprows=1)
x = dados[:, 0]     # usuários simultâneos
y = dados[:, 1]     # latência média (ms)

**✍️ Passo 1.** Desenhe `y` contra `x` com bolinhas (`"o"`), com nomes nos eixos.

In [ ]:
# ✍️ passo 1

**Preveja:** os pontos estão em cima de uma reta?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: seguem uma reta, mas espalhados em volta dela. Um polinômio de grau 24 que
passasse por todos seria absurdo — passaria pelo ruído.

📖 [capítulo 13 · Dados com ruído](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#dados-com-ruido)

</details>

## 2. No quadro: mínimos quadrados

📖 [capítulo 13 · No quadro: mínimos quadrados](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#no-quadro-minimos-quadrados)

### 🧑‍🏫 No quadro — a reta de mínimos quadrados

Caderno de papel aberto. No quadro:

1. o resíduo de cada ponto, $e_i = y_i - (a_0 + a_1 x_i)$;
2. por que somar os **quadrados** dos resíduos;
3. o mínimo: as duas derivadas parciais iguais a zero (capítulo 6);
4. as equações normais — um sistema 2 × 2 (capítulo 7);
5. à mão, com 4 pontos.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$
\begin{bmatrix} n & \sum x_i \\ \sum x_i & \sum x_i^2 \end{bmatrix}
\begin{bmatrix} a_0 \\ a_1 \end{bmatrix} =
\begin{bmatrix} \sum y_i \\ \sum x_i y_i \end{bmatrix}
$$

Com $(1, 2), (2, 3), (3, 5), (4, 6)$: $a_0 = 0{,}5$ e $a_1 = 1{,}4$.

</details>

**✍️ Passo 2.** Calcule as quatro somas das equações normais (`n = len(x)`, `np.sum(x)`, `np.sum(x**2)`, `np.sum(y)`, `np.sum(x * y)`), monte a matriz 2 × 2 e o lado direito, e resolva com `a0, a1 = np.linalg.solve(A, b)`.

In [ ]:
# ✍️ passo 2

**Preveja:** quanto a latência cresce por usuário a mais?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`a1 = 0.079` ms por usuário, sobre uma base de `a0 = 12.8` ms.

</details>

**✍️ Passo 3.** Desenhe os pontos e a reta `a0 + a1 * x` na mesma figura, e calcule a previsão para 600 usuários.

In [ ]:
# ✍️ passo 3

**Preveja:** a previsão para 600 usuários é confiável?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

60 ms. Razoavelmente confiável: 600 está perto da faixa medida (20 a 500). Para
5000 usuários, seria extrapolação — o servidor pode travar muito antes.

📖 [capítulo 13 · No quadro: mínimos quadrados](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#no-quadro-minimos-quadrados)

</details>

### 🎯 Sua vez — Os resíduos

Escreva `residuos(x, y, a0, a1)`, que devolve o array dos resíduos $y_i - (a_0 + a_1x_i)$.

In [ ]:
def residuos(x, y, a0, a1):
    # sua solução aqui
    pass

In [ ]:
confere(residuos, [
    ((np.array([1.0, 2.0, 3.0, 4.0]), np.array([2.0, 3.0, 5.0, 6.0]), 0.5, 1.4), [0.10000000000000009, -0.2999999999999998, 0.3000000000000007, -0.09999999999999964]),
], tol=1e-9)

<details>
<summary><b>💡 Dica</b></summary>

Uma linha: com arrays, a conta vale para todos os elementos de uma vez.

</details>

## 3. Quão boa é a reta

$R^2 = 1 - \dfrac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}$: quanto da variação
de $y$ a reta explica, comparada com "chutar a média".

📖 [capítulo 13 · Quão boa é a reta](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#quao-boa-e-a-reta)

> 🧰 **Comando novo: `np.mean`**
>
> `np.mean(a)` é a média dos elementos: a soma dividida pela quantidade.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
notas = np.array([6.5, 8.0, 7.5, 9.0])
print(np.mean(notas), np.sum(notas) / len(notas))

**✍️ Passo 4.** Calcule o $R^2$ da reta da latência.

In [ ]:
# ✍️ passo 4

**Preveja:** a reta explica mais ou menos que 90 % da variação?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`0.953`: 95 %. O resto é o ruído das medições.

📖 [capítulo 13 · Quão boa é a reta](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#quao-boa-e-a-reta)

</details>

### 🎯 Sua vez — O espalhamento em volta da reta

Escreva `desvio_residuos(x, y, a0, a1)`, que devolve $\sqrt{\sum e_i^2 / (n - 2)}$: o tamanho típico do erro da reta, na unidade de $y$.

In [ ]:
def desvio_residuos(x, y, a0, a1):
    # sua solução aqui
    pass

In [ ]:
confere(desvio_residuos, [
    ((np.array([1.0, 2.0, 3.0, 4.0]), np.array([2.0, 3.0, 5.0, 6.0]), 0.5, 1.4), 0.31622776601683816),
])

<details>
<summary><b>💡 Dica</b></summary>

Os resíduos, a soma dos quadrados, a divisão por `len(x) - 2` e a raiz.

</details>

## 4. Os resíduos contam o resto da história

📖 [capítulo 13 · Os resíduos contam o resto da história](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#os-residuos-contam-o-resto-da-historia)

In [ ]:
# 📦 dados prontos — só rode esta célula
# CO2 na atmosfera (ppm), médias anuais de Mauna Loa, de 5 em 5 anos.
dados_co2 = np.loadtxt("co2_mauna_loa.csv", delimiter=",", skiprows=1)
ano = dados_co2[:, 0]
co2 = dados_co2[:, 1]

**✍️ Passo 5.** Ajuste uma reta a `co2` × `ano` (com as equações normais ou com o que você já fez), calcule o $R^2$ e desenhe os **resíduos** contra o ano, com `plt.axhline(0, color="black")`.

In [ ]:
# ✍️ passo 5

**Preveja:** com $R^2$ acima de 0,98, os resíduos vão parecer ruído?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: formam um **U** — positivos nas pontas, negativos no meio. O CO₂ não cresce
em ritmo constante: **acelera**. Um $R^2$ alto não garante o modelo certo; o
gráfico dos resíduos, sim.

📖 [capítulo 13 · Os resíduos contam o resto da história](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#os-residuos-contam-o-resto-da-historia)

</details>

> ⚠️ **Armadilha.** Olhar só o $R^2$ é o erro mais comum em regressão. Resíduos com padrão (curva,
funil, ondas) querem dizer que falta algo no modelo.

## 5. Confira com a biblioteca

📖 [capítulo 13 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#confira-com-a-biblioteca)

> 🧰 **Comando novo: `np.polyfit`**
>
> `np.polyfit(x, y, g)` devolve os coeficientes do polinômio de grau `g` de mínimos
> quadrados, **do grau mais alto para o mais baixo**: para uma reta,
> `[inclinação, intercepto]`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.polyfit([1, 2, 3, 4], [2, 3, 5, 6], 1))

**✍️ Passo 6.** Confira a reta da latência com `np.polyfit(x, y, 1)`.

In [ ]:
# ✍️ passo 6

**Preveja:** os números batem com `a0` e `a1`? Em que ordem?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Batem, na ordem **inversa**: `[0.079, 12.765]`. Trocar inclinação e intercepto
é o erro clássico com o `polyfit`.

📖 [capítulo 13 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#confira-com-a-biblioteca)

</details>

## 6. Mesmo método, outra área

**Esporte.** O recorde mundial dos 100 m rasos desde 1968 (`recordes_100m.csv`: ano,
tempo em segundos).

📖 [capítulo 13 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Leia o arquivo, ajuste uma reta com `np.polyfit` e calcule a previsão para 2030 e o ano em que a reta daria 0 segundo.

In [ ]:
# ✍️ passo 7

**Preveja:** as duas previsões fazem sentido?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

2030: 9,54 s — razoável, perto dos dados. Zero segundo no ano **3229** —
absurdo. Fora da faixa dos dados, a reta não sabe que existe um limite físico.

📖 [capítulo 13 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#mesmo-metodo-outra-area)

</details>

> ⚠️ **Correlação não é causalidade.** Uma reta bem ajustada diz que $x$ e $y$ andam
> **juntos**, e não que um causa o outro: as vendas de sorvete e os afogamentos sobem
> juntos no verão, e nenhum causa o outro — os dois vêm do calor.
>
> 📖 [capítulo 13 · Correlação não é causalidade](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/13-regressao-linear/#correlacao-nao-e-causalidade)

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: usar a reta para prever — e saber o quanto confiar.

## 🧩 Resolvendo o problema

> *"**No ritmo atual, em que ano o CO₂ passa de 450 ppm?**"* — a jornalista.

Com a reta $\hat y = a_0 + a_1 \cdot \text{ano}$, o ano em que ela chega a um limite é
$(\text{limite} - a_0)/a_1$.

### 🎯 Sua vez — O ano do limite

Escreva `ano_do_limite(anos, valores, limite)`, que ajusta uma reta com
`np.polyfit` e devolve o ano em que ela atinge `limite`.

In [ ]:
def ano_do_limite(anos, valores, limite):
    # sua solução aqui
    pass

In [ ]:
confere(ano_do_limite, [
    ((ano, co2, 450), 2047.2778844713218),
    (([2000, 2010], [10.0, 20.0], 25.0), 2015.0),
])

<details>
<summary><b>💡 Dica</b></summary>

`coef = np.polyfit(anos, valores, 1)`; lembre que `coef[0]` é a inclinação e `coef[1]`, o intercepto.

</details>

In [ ]:
resposta = ano_do_limite(ano, co2, 450)
print("pela reta, 450 ppm em:", resposta)

<details>
<summary><b>▶ O que os números dizem</b></summary>

A reta diz **2047**. Mas os resíduos do bloco 4 mostraram que o CO₂ **acelera**,
e a reta, que cresce sempre no mesmo ritmo, fica abaixo dos dados mais recentes (o
último resíduo é +8 ppm). Uma reta, para quem acelera, **atrasa** a previsão.

A resposta honesta para a jornalista é: "antes de 2047". Quanto antes? Para isso
é preciso um modelo que curve — é o capítulo 14, que ajusta um polinômio de grau 2 aos
mesmos dados.

</details>

## 📋 A lista

Abra a [Lista 13](https://lacouth.github.io/metodos_telecom-site/listas/lista13/). O **Exercício 01** é à mão (✏️): uma regressão com 4 pontos.
Comece por ele, no papel.

**a)** Quanto vale $\sum x_i y_i$ para $(1, 2), (2, 3), (3, 5), (4, 6)$?

<details>
<summary><b>▶ Resposta</b></summary>

$2 + 6 + 15 + 24 = 47$.

</details>

Termine o exercício e siga para o **Exercício 02**, a reta como função.

## 🚪 Antes de sair

**1.** Por que somar os **quadrados** dos resíduos, e não os resíduos?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque resíduos positivos e negativos se cancelariam: uma reta péssima poderia ter soma zero. O quadrado deixa todos positivos (e pune mais os erros grandes).

</details>

**2.** O que as equações normais têm a ver com a Unidade 4?

<details>
<summary><b>▶ Resposta da 2</b></summary>

São um sistema linear: para a reta, 2 × 2; para um polinômio de grau $g$, $(g + 1) \times (g + 1)$. Ajustar é resolver um sistema.

</details>

**3.** $R^2 = 0{,}98$ garante que a reta é o modelo certo?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Não: o CO₂ tem $R^2 = 0{,}98$ e resíduos em U. Só o gráfico dos resíduos mostra se sobrou um padrão.

</details>

## 🏠 Para casa

- Refaça no papel a dedução das equações normais **sem olhar**.
- Termine a [Lista 13](https://lacouth.github.io/metodos_telecom-site/listas/lista13/).
- Leia o começo do [capítulo 14](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/14-ajuste-nao-linear/): e se o
  modelo certo for uma curva?